In [12]:
!pip uninstall -y pandas pyarrow
!pip install --no-cache-dir pandas==2.1.4 pyarrow==14.0.2

Found existing installation: pandas 2.1.4
Uninstalling pandas-2.1.4:
  Successfully uninstalled pandas-2.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 377.3 kB/s  0:00:31m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 315.4 kB/s  0:01:54m0:00:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


# **PRODUCT LOCALISATION USING DINOv2**

In [ ]:
import os
import gc
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel

CANDIDATE_ROOTS = [
    Path("/scratch/krishank_iitp/project/data_scratch/data"),
    Path("/scratch/krishank_iitp/project/data/data"),
    Path("/scratch/krishank_iitp/project/data"),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
print("Using ROOT:", ROOT)


MAX_PRODUCTS = None
MAX_ORIGINALS_PER_PRODUCT = None
MAX_REVIEWS_PER_PRODUCT = None

SEARCH_MAX_SIDE = 640
BATCH_SIZE = 32
GLOBAL_SCALES = [0.3, 0.5]
TOPK_GLOBAL = 2

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


DINO_CKPT = "facebook/dinov2-small"

processor = AutoImageProcessor.from_pretrained(DINO_CKPT)
model = AutoModel.from_pretrained(
    DINO_CKPT,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print("Model device:", next(model.parameters()).device)


def is_image(p):
    return p.is_file() and p.suffix.lower() in IMG_EXTS

def find_images(folder):
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if is_image(p)])

from PIL import Image, UnidentifiedImageError

def pil_open(path):
    try:
        img = Image.open(path).convert("RGB")
        return img
    except (UnidentifiedImageError, OSError):
        return None

def clip_box(box,w,h):
    x1,y1,x2,y2 = box
    x1=max(0,min(x1,w-1))
    y1=max(0,min(y1,h-1))
    x2=max(1,min(x2,w))
    y2=max(1,min(y2,h))
    return int(x1),int(y1),int(x2),int(y2)

def draw_overlay(img, box, text=None):
    out = img.copy()
    draw = ImageDraw.Draw(out)
    draw.rectangle(box, outline="red", width=4)
    if text:
        draw.text((box[0],box[1]), text, fill="red")
    return out

def save_mask(path, size_hw, box):
    h,w=size_hw
    mask=np.zeros((h,w),dtype=np.uint8)
    x1,y1,x2,y2=box
    mask[y1:y2,x1:x2]=255
    cv2.imwrite(str(path),mask)


# EMBEDDINGS
@torch.inference_mode()
def embed_batch(images):
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device, dtype=torch.float16) if torch.is_floating_point(v) else v.to(device) for k,v in inputs.items()}
    outputs = model(**inputs)
    hs = outputs.last_hidden_state
    cls = hs[:,0,:]
    patch = hs[:,1:,:].mean(dim=1)
    feat = torch.cat([cls,patch],dim=-1)
    feat = F.normalize(feat,dim=-1)
    return feat

# REGION PROPOSALS
def generate_boxes(w,h,scales):
    boxes=[]
    for s in scales:
        bw=int(w*s)
        bh=int(h*s)
        step=max(16,int(bw*0.5))
        for y in range(0,h-bh+1,step):
            for x in range(0,w-bw+1,step):
                boxes.append((x,y,x+bw,y+bh))
    return boxes

# PRODUCT PROCESSING
def process_product(product_dir):
    original_dir = product_dir / "original_images"
    review_dir = product_dir / "raw_review_images"

    originals = find_images(original_dir)
    reviews = find_images(review_dir)

    if MAX_ORIGINALS_PER_PRODUCT:
        originals = originals[:MAX_ORIGINALS_PER_PRODUCT]
    if MAX_REVIEWS_PER_PRODUCT:
        reviews = reviews[:MAX_REVIEWS_PER_PRODUCT]

    if not originals or not reviews:
        return []

    comp_root = product_dir / "comparisons_dinov2_fast"
    comp_root.mkdir(exist_ok=True)

    # ORIGINAL EMBEDDINGS
    orig_imgs=[]
    valid_originals=[]

    for p in originals:
        img = pil_open(p)
        if img is not None:
            orig_imgs.append(img)
            valid_originals.append(p)
    orig_feats=[]
    for i in range(0,len(orig_imgs),BATCH_SIZE):
        orig_feats.append(embed_batch(orig_imgs[i:i+BATCH_SIZE]))
    orig_feats=torch.cat(orig_feats)

    rows=[]

    for review_path in reviews:
        try:
            out_dir = comp_root / review_path.stem
            if (out_dir / "crop.jpg").exists():
                continue
    
            # now load image
            img_full = pil_open(review_path)
            if img_full is None:
                continue
            w0,h0 = img_full.size

            # resize for search
            scale=1
            if max(w0,h0)>SEARCH_MAX_SIDE:
                scale=SEARCH_MAX_SIDE/max(w0,h0)
                img_search=img_full.resize((int(w0*scale),int(h0*scale)))
            else:
                img_search=img_full.copy()

            w,h=img_search.size

            # GLOBAL SEARCH
            boxes = generate_boxes(w,h,GLOBAL_SCALES)
            crops=[img_search.crop(b) for b in boxes]

            feats=[]
            for i in range(0,len(crops),BATCH_SIZE):
                feats.append(embed_batch(crops[i:i+BATCH_SIZE]))
            feats=torch.cat(feats)

            sim = feats @ orig_feats.T
            best_sim,_ = sim.max(dim=1)
            idx = torch.argmax(best_sim).item()
            best_box = boxes[idx]

            # scale back
            x1,y1,x2,y2 = best_box
            x1=int(x1/scale)
            y1=int(y1/scale)
            x2=int(x2/scale)
            y2=int(y2/scale)
            x1,y1,x2,y2=clip_box((x1,y1,x2,y2),w0,h0)

            # SAVE
            out_dir = comp_root / review_path.stem
            out_dir.mkdir(exist_ok=True)

            crop_path = out_dir/"crop.jpg"
            overlay_path = out_dir/"overlay.jpg"
            mask_path = out_dir/"mask.png"
            pred_path = out_dir/"prediction.json"

            crop = img_full.crop((x1,y1,x2,y2))
            crop.save(crop_path)

            overlay = draw_overlay(img_full,(x1,y1,x2,y2),f"sim")
            overlay.save(overlay_path)

            save_mask(mask_path,img_full.size[::-1],(x1,y1,x2,y2))

            with open(pred_path,"w") as f:
                json.dump({
                    "review":str(review_path),
                    "bbox":[x1,y1,x2,y2]
                },f)

            rows.append({
                "review":str(review_path),
                "bbox":[x1,y1,x2,y2],
                "crop":str(crop_path)
            })

        except Exception as e:
            print("Error:",e)

    pd.DataFrame(rows).to_csv(comp_root/"results.csv",index=False)
    return rows


# FIND PRODUCTS
product_dirs=[]
for p in ROOT.rglob("original_images"):
    product_dirs.append(p.parent)

if MAX_PRODUCTS:
    product_dirs=product_dirs[:MAX_PRODUCTS]

print("Products:",len(product_dirs))

# RUN
all_rows=[]
for p in tqdm(product_dirs):
    all_rows.extend(process_product(p))
    cleanup()

print("DONE")

/home/krishank_iitp/.conda/envs/gpu_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using ROOT: /scratch/krishank_iitp/project/data/data
Using device: cuda
GPU: NVIDIA A100 80GB PCIe


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████████████████████████████████████████████| 223/223 [00:01<00:00, 138.41it/s]


Model device: cuda:0
Products: 3949


100%|█████████████████████████████████████████████████████████████████████████████| 3949/3949 [1:20:52<00:00,  1.23s/it]

DONE


# **DINOv2 V/S ANNOTATED**

In [ ]:
from pathlib import Path
import os
import json
import csv
import gc
import numpy as np
import cv2
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm


CANDIDATE_ROOTS = [
    Path("/scratch/krishank_iitp/project/data_scratch/data"),
    Path("/scratch/krishank_iitp/project/data/data"),
    Path("/scratch/krishank_iitp/project/data"),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
print("Using ROOT:", ROOT)

PRED_DIR_NAME = "comparisons_dinov2_fast"
GT_DIR_NAMES = ["annotated_defective", "annotated_non_defective"]

OUT_ROOT = ROOT / "mask_eval_results"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

IOU_THRESHOLDS = [0.50, 0.75]
COCO_THRESHOLDS = [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)]

# Helpers
def safe_open_mask(path: Path):
    try:
        img = Image.open(path).convert("RGBA" if path.suffix.lower() == ".png" else "RGB")
        return img
    except (UnidentifiedImageError, OSError, IOError):
        return None

def find_product_dirs():
    product_dirs = []
    for p in ROOT.rglob("original_images"):
        if p.is_dir():
            product_dirs.append(p.parent)
    return sorted(set(product_dirs))

def rel_under_raw_review(review_path: Path):
    parts = review_path.parts
    if "raw_review_images" in parts:
        idx = parts.index("raw_review_images")
        return Path(*parts[idx + 1:])
    return Path(review_path.name)

def load_prediction_items(product_dir: Path):
    pred_root = product_dir / PRED_DIR_NAME
    if not pred_root.exists():
        return []

    items = []
    for pj in pred_root.rglob("prediction.json"):
        try:
            with open(pj, "r", encoding="utf-8") as f:
                data = json.load(f)

            bbox = data.get("bbox", None)
            review_path = data.get("review_path", "")
            score = float(data.get("sim", data.get("combined_score", 0.0)))

            mask_path = pj.parent / "mask.png"
            if bbox is None or len(bbox) != 4 or not mask_path.exists():
                continue

            items.append({
                "prediction_json": str(pj),
                "mask_path": str(mask_path),
                "review_path": review_path,
                "bbox": tuple(map(int, bbox)),
                "score": score,
            })
        except Exception:
            continue

    return items

def find_gt_path(product_dir: Path, review_path_str: str):
    review_path = Path(review_path_str)
    rel = rel_under_raw_review(review_path)

    gt_roots = [product_dir / d for d in GT_DIR_NAMES if (product_dir / d).exists()]
    if not gt_roots:
        return None, None

    # 1) exact relative path match
    for gt_root in gt_roots:
        candidate = gt_root / rel
        if candidate.exists() and candidate.is_file():
            return candidate, gt_root.name

    # 2) same stem anywhere in GT folder
    stem = review_path.stem
    for gt_root in gt_roots:
        matches = [p for p in gt_root.rglob(f"{stem}.*") if p.is_file()]
        if matches:
            return matches[0], gt_root.name

    return None, None

def mask_from_pred(path: Path):
    img = safe_open_mask(path)
    if img is None:
        return None

    arr = np.array(img)

    # PNG mask saved by your pipeline: white rectangle on black background
    if arr.ndim == 3 and arr.shape[2] == 4:
        alpha = arr[:, :, 3]
        mask = alpha > 0
    elif arr.ndim == 3:
        gray = cv2.cvtColor(arr[:, :, :3], cv2.COLOR_RGB2GRAY)
        mask = gray > 0
    else:
        mask = arr > 0

    if mask.sum() == 0:
        return None
    return mask.astype(bool)

def mask_from_gt(path: Path):
    img = cv2.imread(str(path))

    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Only keep BRIGHT WHITE pixels (annotation)
    _, mask = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)

    mask = mask > 0

    # sanity check
    if mask.sum() < 50:
        return None

    return mask

def resize_mask(mask, target_hw):
    th, tw = target_hw
    mh, mw = mask.shape[:2]
    if (mh, mw) == (th, tw):
        return mask
    resized = cv2.resize(mask.astype(np.uint8), (tw, th), interpolation=cv2.INTER_NEAREST)
    return resized.astype(bool)

def iou_mask(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 0.0
    return float(inter / union)

def coverage_scores(pred_mask, gt_mask):
    inter = np.logical_and(pred_mask, gt_mask).sum()
    pred_area = pred_mask.sum()
    gt_area = gt_mask.sum()
    pred_cover = float(inter / pred_area) if pred_area > 0 else 0.0
    gt_cover = float(inter / gt_area) if gt_area > 0 else 0.0
    return pred_cover, gt_cover

def average_precision(recalls, precisions):
    if len(recalls) == 0:
        return 0.0
    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    xs = np.linspace(0, 1, 101)
    ap = 0.0
    for x in xs:
        vals = mpre[mrec >= x]
        ap += float(np.max(vals)) if len(vals) > 0 else 0.0
    return ap / 101.0

def save_csv(rows, path: Path, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# Main evaluation
product_dirs = find_product_dirs()
print("Product folders found:", len(product_dirs))

all_rows = []
summary_rows = []

for product_dir in tqdm(product_dirs, desc="Evaluating masks"):
    preds = load_prediction_items(product_dir)
    if not preds:
        continue

    gt_exists = any((product_dir / d).exists() for d in GT_DIR_NAMES)
    if not gt_exists:
        continue

    valid_rows = []

    for item in preds:
        review_path = Path(item["review_path"])
        pred_mask_path = Path(item["mask_path"])
        gt_path, gt_folder = find_gt_path(product_dir, item["review_path"])

        pred_mask = mask_from_pred(pred_mask_path)
        if pred_mask is None:
            continue

        gt_mask = None
        if gt_path is not None:
            gt_mask = mask_from_gt(gt_path)

        if gt_mask is None:
            all_rows.append({
                "category": product_dir.relative_to(ROOT).parts[0] if len(product_dir.relative_to(ROOT).parts) > 0 else "",
                "product_id": product_dir.name,
                "review_path": str(review_path),
                "pred_mask_path": str(pred_mask_path),
                "gt_path": str(gt_path) if gt_path else "",
                "gt_folder": gt_folder if gt_path else "",
                "score": item["score"],
                "iou": np.nan,
                "pred_cover": np.nan,
                "gt_cover": np.nan,
                "containment_hit": 0,
                "status": "gt_missing_or_unreadable",
            })
            continue
        
        # match GT size
        gt_h, gt_w = gt_mask.shape[:2]
        pred_mask = resize_mask(pred_mask, (gt_h, gt_w))

        iou = iou_mask(pred_mask, gt_mask)
        pred_cover, gt_cover = coverage_scores(pred_mask, gt_mask)

        # "containment" = most of GT lies inside predicted rectangle
        containment_hit = 1 if gt_cover >= 0.80 else 0

        row = {
            "category": product_dir.relative_to(ROOT).parts[0] if len(product_dir.relative_to(ROOT).parts) > 0 else "",
            "product_id": product_dir.name,
            "review_path": str(review_path),
            "pred_mask_path": str(pred_mask_path),
            "gt_path": str(gt_path) if gt_path else "",
            "gt_folder": gt_folder if gt_path else "",
            "score": item["score"],
            "iou": float(iou),
            "pred_cover": float(pred_cover),
            "gt_cover": float(gt_cover),
            "containment_hit": int(containment_hit),
            "status": "ok",
        }
        all_rows.append(row)
        valid_rows.append(row)

    if not valid_rows:
        continue

    mean_iou = float(np.mean([r["iou"] for r in valid_rows]))
    mean_gt_cover = float(np.mean([r["gt_cover"] for r in valid_rows]))
    containment_rate = float(np.mean([r["containment_hit"] for r in valid_rows]))

    for thr in IOU_THRESHOLDS:
        tp = sum(1 for r in valid_rows if r["iou"] >= thr)
        fp = sum(1 for r in valid_rows if r["iou"] < thr)
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (len(valid_rows) + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)

        summary_rows.append({
            "category": product_dir.relative_to(ROOT).parts[0] if len(product_dir.relative_to(ROOT).parts) > 0 else "",
            "product_id": product_dir.name,
            "metric": f"IoU@{thr:.2f}",
            "num_samples": len(valid_rows),
            "mean_iou": mean_iou,
            "mean_gt_cover": mean_gt_cover,
            "containment_rate": containment_rate,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

# Global AP / mAP
evaluated = [r for r in all_rows if r.get("status") == "ok"]
ap_table = []

if evaluated:
    scored = sorted(evaluated, key=lambda x: float(x["score"]), reverse=True)

    for thr in COCO_THRESHOLDS:
        tp = np.array([1 if float(r["iou"]) >= thr else 0 for r in scored], dtype=np.float32)
        fp = 1.0 - tp

        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)

        precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
        recalls = tp_cum / max(len(scored), 1)

        ap = average_precision(recalls, precisions)
        ap_table.append({
            "iou_thr": thr,
            "ap": ap,
        })

    mAP = float(np.mean([x["ap"] for x in ap_table]))
    ap50 = next(x["ap"] for x in ap_table if abs(x["iou_thr"] - 0.50) < 1e-9)
    ap75 = next(x["ap"] for x in ap_table if abs(x["iou_thr"] - 0.75) < 1e-9)
    mean_iou_global = float(np.mean([r["iou"] for r in evaluated]))
    mean_gt_cover_global = float(np.mean([r["gt_cover"] for r in evaluated]))
    containment_global = float(np.mean([r["containment_hit"] for r in evaluated]))
else:
    mAP = ap50 = ap75 = mean_iou_global = mean_gt_cover_global = containment_global = 0.0

all_csv = OUT_ROOT / "mask_eval_all.csv"
summary_csv = OUT_ROOT / "mask_eval_summary.csv"
ap_csv = OUT_ROOT / "mask_eval_ap_table.csv"

save_csv(all_rows, all_csv, [
    "category", "product_id", "review_path", "pred_mask_path",
    "gt_path", "gt_folder", "score", "iou",
    "pred_cover", "gt_cover", "containment_hit", "status"
])

save_csv(summary_rows, summary_csv, [
    "category", "product_id", "metric", "num_samples",
    "mean_iou", "mean_gt_cover", "containment_rate",
    "precision", "recall", "f1"
])

if ap_table:
    save_csv(ap_table, ap_csv, ["iou_thr", "ap"])

print("\nSaved:")
print(all_csv)
print(summary_csv)
if ap_table:
    print(ap_csv)

print("\nGLOBAL SUMMARY ")
print("Evaluated images:", len(evaluated))
print(f"Mean IoU: {mean_iou_global:.4f}")
print(f"Mean GT coverage: {mean_gt_cover_global:.4f}")
print(f"Containment rate (GT coverage >= 0.80): {containment_global:.4f}")
print(f"AP@0.50: {ap50:.4f}")
print(f"AP@0.75: {ap75:.4f}")
print(f"mAP@0.50:0.95: {mAP:.4f}")

Using ROOT: /scratch/krishank_iitp/project/data/data
Product folders found: 3949


Evaluating masks: 100%|█████████████████████████████████████████████████████████████| 3949/3949 [14:10<00:00,  4.64it/s]



Saved:
/scratch/krishank_iitp/project/data/data/mask_eval_results/mask_eval_all.csv
/scratch/krishank_iitp/project/data/data/mask_eval_results/mask_eval_summary.csv
/scratch/krishank_iitp/project/data/data/mask_eval_results/mask_eval_ap_table.csv

==== GLOBAL SUMMARY ====
Evaluated images: 505
Mean IoU: 0.3229
Mean GT coverage: 1.0000
Containment rate (GT coverage >= 0.80): 1.0000
AP@0.50: 0.0358
AP@0.75: 0.0012
mAP@0.50:0.95: 0.0078
